# Predicting Review Popularity (Amazon Kindle book reviews) - Random Forest
UE24CS352A ML mini-project, adapted from *Text-Based Prediction of Book Review Popularity* (B. Daly) to the Kaggle **Amazon Kindle Book Review** dataset (`all_kindle_review .csv`, ~12k reviews).

A random forest (bagged ensemble of decision trees with random feature subsets). `max_depth` and `min_samples_leaf`
are tuned on the validation set; the number of trees is fixed at 200.

**Task:** binary classification - is a review *popular* (>= a cutoff of helpful votes, chosen so ~25% are popular)?

**Feature subsets:** A = 4 non-textual · B = 12 non-textual + textual · C = B + Bag-of-Words · D = B + TF-IDF.

**Run:** upload `all_kindle_review .csv` to `MyDrive/kindle/`, then *Runtime -> Run all*.

## 1. Setup

In [ ]:
!pip -q install orjson

In [ ]:
import os, re, gzip, json, zlib, time, warnings, subprocess
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

try:
    import orjson
    loads = orjson.loads
except ImportError:
    loads = json.loads

# Google Drive is used to keep the big data files + the cached features between sessions
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/kindle"
except ImportError:                      # running locally instead of Colab
    DATA_DIR = "./kindle"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "outputs"), exist_ok=True)
print("Data directory:", DATA_DIR)

In [ ]:
# ------------------------------- CONFIG -------------------------------
CSV_PATTERN = "all_kindle_review*.csv"   # Kaggle file is named "all_kindle_review .csv" (note the space) - both work
HELPFUL_MIN = None        # popular = helpful_votes >= HELPFUL_MIN.  None = pick automatically so ~25% of reviews are popular
TARGET_PREVALENCE = 0.25
MAX_REVIEWS = None        # None = use all (~12k reviews)
POS_TOKEN_LIMIT = 150
VOCAB_SIZE = 2000
RUN_TEXT_SUBSETS = True   # also run subsets C (BOW) and D (TF-IDF)
REBUILD_CACHE = False
SEED = 42
CACHE_FILE = os.path.join(DATA_DIR, "kindle_features.pkl")

## 2. Locate the data file

In [ ]:
import glob
matches = glob.glob(os.path.join(DATA_DIR, CSV_PATTERN))
if not matches and (not os.path.exists(CACHE_FILE) or REBUILD_CACHE):
    raise FileNotFoundError(
        f"Put the Kaggle file all_kindle_review .csv in {DATA_DIR}\n"
        "(kaggle.com/datasets/meetnagadia/amazon-kindle-book-review-for-sentiment-analysis -> Download -> unzip -> upload "
        "to that Drive folder). Use all_kindle_review .csv, NOT preprocessed_kindle_review.csv")
csv_path = matches[0] if matches else None
print("Using:", csv_path)

## 3. Feature engineering functions

In [ ]:
import nltk
for pkg in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng", "vader_lexicon"]:
    nltk.download(pkg, quiet=True)
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from multiprocessing import Pool

sia = SentimentIntensityAnalyzer()

NON_TEXT = ["user_reviews", "days_since_review", "user_rating", "rating_diff"]
TEXT     = ["num_words", "avg_sent_len", "avg_word_len", "pct_nouns", "pct_verbs", "pct_adj_adv", "quote", "sentiment"]
B_COLS   = NON_TEXT + TEXT                       # 4 + 8 = 12 features (subset B)

EN_COMMON = set("the a an and or but of to in on for with is was are were be been it this that i you he she they we "
                "my his her their its as at by from not have has had do did so if what who which will would can could "
                "just very".split())

def clean_html(t):
    t = re.sub(r"<[^>]+>", " ", t)               # reviews contain <br /> tags etc.
    return re.sub(r"\s+", " ", t).strip()

def looks_english(t):
    """Cheap stand-in for the paper's fastText language-ID filter."""
    if not t: return False
    ascii_ratio = len(t.encode("ascii", "ignore")) / len(t)
    toks = re.findall(r"[a-z']+", t.lower())
    if not toks or ascii_ratio < 0.95: return False
    if len(toks) >= 6:
        return sum(w in EN_COMMON for w in toks) / len(toks) >= 0.10
    return True

def text_feats(text):
    """num_words, avg_sent_len, avg_word_len, pct_nouns, pct_verbs, pct_adj_adv, quote, sentiment"""
    quote = int('"' in text or "\u201c" in text)
    sentiment = sia.polarity_scores(text)["compound"]
    sents = sent_tokenize(text)
    toks  = word_tokenize(text)
    words = [w for w in toks if w.isalpha()]
    n = len(words)
    if n == 0:
        return (0, 0.0, 0.0, 0.0, 0.0, 0.0, quote, sentiment)
    tags = [t for w, t in nltk.pos_tag(toks[:POS_TOKEN_LIMIT]) if w.isalpha()]
    m = max(len(tags), 1)
    nouns = sum(t.startswith("NN") for t in tags) / m
    verbs = sum(t.startswith("VB") for t in tags) / m
    advadj = sum(t.startswith("JJ") or t.startswith("RB") for t in tags) / m
    return (n, n / max(len(sents), 1), sum(len(w) for w in words) / n, nouns, verbs, advadj, quote, sentiment)

## 4. Build the labelled dataset (or load it from the cache)

In [ ]:
import ast
if os.path.exists(CACHE_FILE) and not REBUILD_CACHE:
    df = pd.read_pickle(CACHE_FILE)
    print("Loaded cached features:", CACHE_FILE, df.shape)
else:
    raw = pd.read_csv(csv_path)
    raw.columns = [c.strip() for c in raw.columns]
    need = ["asin", "helpful", "rating", "reviewText", "reviewerID", "unixReviewTime"]
    missing = [c for c in need if c not in raw.columns]
    assert not missing, f"Missing columns {missing}; file has {list(raw.columns)}"

    h = raw.helpful.map(ast.literal_eval)               # "[8, 10]" -> [8, 10]
    df = pd.DataFrame({"book_id": raw.asin, "user_id": raw.reviewerID, "user_rating": raw.rating,
                       "helpful_votes": h.str[0], "total_votes": h.str[1],
                       "unix": raw.unixReviewTime, "review_text": raw.reviewText.fillna("").astype(str)})
    print(f"Raw reviews: {len(df):,} | share with 0 helpful votes: {(df.helpful_votes == 0).mean():.2f}")

    # features that need the whole file
    df["user_reviews"] = df.user_id.map(df.user_id.value_counts())
    df["days_since_review"] = (df.unix.max() - df.unix) / 86400.0
    df["rating_diff"] = (df.user_rating - df.groupby("book_id")["user_rating"].transform("mean")).abs()

    # label: popular = enough helpful votes
    t = HELPFUL_MIN
    if t is None:
        t = 1
        while (df.helpful_votes >= t).mean() > TARGET_PREVALENCE: t += 1
    df["popular"] = (df.helpful_votes >= t).astype(int)
    df["likes_share"] = df.helpful_votes            # continuous popularity signal used in the correlation check
    print(f"Popular = helpful_votes >= {t}  ->  prevalence {df.popular.mean():.3f}")

    df["clean_text"] = df.review_text.map(clean_html)
    df = df[(df.clean_text.str.len() > 0) & df.clean_text.map(looks_english)]
    if MAX_REVIEWS and len(df) > MAX_REVIEWS: df = df.sample(MAX_REVIEWS, random_state=SEED)
    df = df.reset_index(drop=True)
    print(f"Reviews used for modelling: {len(df):,}")

    with Pool(os.cpu_count()) as pool:
        feats = list(tqdm(pool.imap(text_feats, df.clean_text.tolist(), chunksize=200),
                          total=len(df), desc="text features"))
    df[TEXT] = pd.DataFrame(feats, columns=TEXT, index=df.index)
    df = df.drop(columns=["review_text"]).dropna(subset=B_COLS).reset_index(drop=True)
    df.to_pickle(CACHE_FILE)
    print("Saved cache ->", CACHE_FILE)

print(df.shape, "| popular prevalence:", round(df.popular.mean(), 3))
df[B_COLS + ["helpful_votes", "popular"]].describe().T.round(3)

## 5. Train / validation / test split (70 / 15 / 15) and text representations

In [ ]:
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

y = df.popular.values
idx = np.arange(len(df))
idx_trval, idx_te = train_test_split(idx, test_size=0.15, stratify=y, random_state=SEED)
idx_tr, idx_val   = train_test_split(idx_trval, test_size=0.15 / 0.85, stratify=y[idx_trval], random_state=SEED)
y_tr, y_val, y_te = y[idx_tr], y[idx_val], y[idx_te]
print(f"train {len(idx_tr):,} | validation {len(idx_val):,} | test {len(idx_te):,}")
print(f"always-'unpopular' baseline accuracy on test: {1 - y_te.mean():.3f}")

# Text representations are fitted on the training split only (no leakage)
vec_kw = dict(stop_words="english", max_features=VOCAB_SIZE, min_df=5,
              token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b", lowercase=True)
txt_tr = df.clean_text.values[idx_tr]
bow   = CountVectorizer(**vec_kw).fit(txt_tr)
tfidf = TfidfVectorizer(**vec_kw).fit(txt_tr)

_cache = {}
def build_X(subset, ids):
    if subset == "A": return df[NON_TEXT].values[ids].astype(np.float32)
    base = df[B_COLS].values[ids].astype(np.float32)
    if subset == "B": return base
    vec = bow if subset == "C" else tfidf
    M = vec.transform(df.clean_text.values[ids])
    return sparse.hstack([sparse.csr_matrix(base), M], format="csr", dtype=np.float32)

def get_data(subset):
    if subset not in _cache:
        _cache[subset] = tuple(build_X(subset, ids) for ids in (idx_tr, idx_val, idx_te))
    return _cache[subset]

def feature_names(subset):
    if subset == "A": return NON_TEXT
    if subset == "B": return B_COLS
    vec = bow if subset == "C" else tfidf
    return B_COLS + [("bow_" if subset == "C" else "tfidf_") + w for w in vec.get_feature_names_out()]

## 6. Model and evaluation helpers

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, roc_curve

def undersample_idx(y_):
    """Balance classes by randomly dropping majority-class (unpopular) rows - as in the paper."""
    rng = np.random.RandomState(SEED)
    pos, neg = np.where(y_ == 1)[0], np.where(y_ == 0)[0]
    sel = np.concatenate([pos, rng.choice(neg, size=len(pos), replace=False)])
    rng.shuffle(sel)
    return sel

def evaluate(proba, y_true, thr=0.5):
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return dict(Accuracy=(tp + tn) / len(y_true), Sensitivity=tp / (tp + fn),
                Specificity=tn / (tn + fp), ROC_AUC=roc_auc_score(y_true, proba))

STORE, ROWS = {}, []

def run_experiment(model_name, subset, undersample, build, grid, fit=None, describe=None):
    """Tune on the validation set (best ROC AUC), report metrics on the untouched test set."""
    Xtr, Xval, Xte = get_data(subset)
    if undersample:
        sel = undersample_idx(y_tr); Xfit, yfit = Xtr[sel], y_tr[sel]
    else:
        Xfit, yfit = Xtr, y_tr
    best = None
    for params in ParameterGrid(grid):
        m = build(params)
        if fit: fit(m, Xfit, yfit, Xval, y_val)
        else:   m.fit(Xfit, yfit)
        val_auc = roc_auc_score(y_val, m.predict_proba(Xval)[:, 1])
        if best is None or val_auc > best[0]:
            best = (val_auc, params, m)
    val_auc, params, m = best
    proba = m.predict_proba(Xte)[:, 1]
    res = evaluate(proba, y_te)
    row = dict(Subset=subset, Model=model_name, UnderSample="Yes" if undersample else "No",
               Hyperparameters=describe(m, params) if describe else str(params),
               **{k: round(v, 3) for k, v in res.items()}, Val_AUC=round(val_auc, 3))
    ROWS.append(row)
    STORE[(subset, undersample)] = dict(model=m, proba=proba)
    print(f"[{subset} | undersample={undersample}] acc={res['Accuracy']:.3f} sens={res['Sensitivity']:.3f} "
          f"spec={res['Specificity']:.3f} auc={res['ROC_AUC']:.3f}   {row['Hyperparameters']}")

EXPERIMENTS = [("A", False), ("B", False), ("A", True), ("B", True)]
if RUN_TEXT_SUBSETS:
    EXPERIMENTS += [("C", True), ("D", True)]

In [ ]:
from sklearn.ensemble import RandomForestClassifier
MODEL_NAME, MODEL_TAG = "Random Forest", "random_forest"
N_TREES = 200

def grid_for(subset):
    if subset in ("C", "D"):     # ~2,000 sparse features -> keep the grid small so it finishes quickly
        return {"max_depth": [10, 16], "min_samples_leaf": [5]}
    return {"max_depth": [8, 12, None], "min_samples_leaf": [1, 5, 20]}

def build(params):
    return RandomForestClassifier(n_estimators=N_TREES, max_features="sqrt", n_jobs=-1,
                                  random_state=SEED, **params)

## 7. Train, tune on validation set, evaluate on test set

In [ ]:
for subset, us in EXPERIMENTS:
    run_experiment(MODEL_NAME, subset, us, build, grid_for(subset))

## 8. Results table

In [ ]:
results = pd.DataFrame(ROWS)
results.to_csv(os.path.join(DATA_DIR, "outputs", f"results_{MODEL_TAG}.csv"), index=False)
print(f"Majority-class baseline accuracy: {1 - y_te.mean():.3f}\n")
results.drop(columns=["Val_AUC"])

## 9. Plots

In [ ]:
plt.figure(figsize=(6, 5))
for (subset, us), d in STORE.items():
    fpr, tpr, _ = roc_curve(y_te, d["proba"])
    plt.plot(fpr, tpr, label=f"{subset} ({'under' if us else 'full'}) AUC={roc_auc_score(y_te, d['proba']):.2f}")
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("False positive rate (1 - specificity)"); plt.ylabel("True positive rate (sensitivity)")
plt.title(f"ROC curves - {MODEL_NAME} (test set)"); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(DATA_DIR, "outputs", f"roc_{MODEL_TAG}.png"), dpi=150); plt.show()

In [ ]:
def plot_importance(subset, us, top=15):
    d = STORE[(subset, us)]
    imp = pd.Series(d["model"].feature_importances_, index=feature_names(subset)).sort_values().tail(top)
    imp.plot.barh(figsize=(6, 0.35 * top + 1.5), color="#4c72b0")
    plt.title(f"{MODEL_NAME}: top {top} feature importances (subset {subset}, {'undersampled' if us else 'full'})")
    plt.tight_layout(); plt.savefig(os.path.join(DATA_DIR, "outputs", f"importance_{MODEL_TAG}_{subset}.png"), dpi=150); plt.show()

plot_importance("B", True)                          # 12 interpretable features
if RUN_TEXT_SUBSETS: plot_importance("D", True)     # incl. TF-IDF words

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
d = STORE[("B", True)]
ConfusionMatrixDisplay.from_predictions(y_te, (d["proba"] >= 0.5).astype(int),
        display_labels=["unpopular", "popular"], cmap="Blues")
plt.title(f"{MODEL_NAME} - subset B, undersampled (test set)")
plt.savefig(os.path.join(DATA_DIR, "outputs", f"confusion_{MODEL_TAG}.png"), dpi=150); plt.show()

# Paper section 5: does predicted probability track the *continuous* likes share?
ls_te = df.likes_share.values[idx_te]
print("Pearson corr(predicted prob, likes_share), subset B undersampled:",
      round(np.corrcoef(d["proba"], ls_te)[0, 1], 3), "  ")